# 1) Pull the data

Read the local preprocessed CSV. Target: **MFBTU (kBtu)**. Keep the existing training, validation and test splits and preprocessing. Use the 10 original numeric predictors.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'cbecs2018_processed.csv').exists())
df = pd.read_csv(root / 'cbecs2018_processed.csv')
features = ['SQFT', 'FLCEILHT', 'WKHRS', 'NWKER', 'MONUSE',
            'HDD65', 'CDD65', 'HEATP', 'COOLP', 'OCCUPYP']
parts = {s: df.loc[df['split'].eq(s)] for s in ['train', 'validation', 'test']}
train = parts['train']
assert df.PUBID.is_unique
assert [len(parts[s]) for s in parts] == [4449, 954, 954]
assert np.isfinite(df[features + ['MFBTU']].to_numpy()).all()
display(df[['PUBID', 'split', 'MFBTU'] + features].head())
display(df.groupby('split').size().rename('Buildings').to_frame())

def evaluate(model, columns):
    rows = []
    for split, data in parts.items():
        pred = model.predict(data[columns])
        rows.append({'split': split, 'R2': r2_score(data.MFBTU, pred),
                     'MAE_kBtu': mean_absolute_error(data.MFBTU, pred),
                     'RMSE_kBtu': np.sqrt(mean_squared_error(data.MFBTU, pred))})
    return pd.DataFrame(rows).set_index('split')


   PUBID  split     MFBTU    SQFT  FLCEILHT  WKHRS  NWKER  MONUSE  HDD65  \
0      1  train  29727152  210000        12     70    350      12   4463   
1      2  train   1730655   28000         8     45     12      12   2424   
2      3  train     52387    2100         9     48      6      12   3218   
3      4  train   3185775  240000        25     55     34      12   1045   
4      5  train   4677009  295000        30     80     10      12   5076   

   CDD65  HEATP  COOLP  OCCUPYP  
0   1759    100    100      100  
1    189     80    100      100  
2   2403    100    100      100  
3   3041      5     45       96  
4   1316    100    100       96  
            Buildings
split                
test              954
train            4449
validation        954


# 2) Single-variable results

Fit each numeric predictor separately using training data. Use HC3 slope tests with Holm-adjusted p < 0.05. Choose the significant predictor with the highest training R², then report R², MAE and RMSE on each split.

In [2]:
rows = []
for feature in features:
    fit = OLS(train.MFBTU, add_constant(train[[feature]])).fit(cov_type='HC3')
    rows.append({'feature': feature, 'training_R2': fit.rsquared,
                 'slope': fit.params[feature], 'p_value_HC3': fit.pvalues[feature]})
screen = pd.DataFrame(rows)
screen['adjusted_p'] = multipletests(screen.p_value_HC3, method='holm')[1]
screen['significant'] = screen.adjusted_p < 0.05
screen = screen.sort_values('training_R2', ascending=False)
display(screen.reset_index(drop=True))

significant_features = screen.loc[screen.significant, 'feature'].tolist()
assert significant_features, 'No predictors passed the significance threshold.'
selected = significant_features[0]
single_model = LinearRegression().fit(train[[selected]], train.MFBTU)
print('Selected single predictor:', selected)
print(f'MFBTU = {single_model.intercept_:.2f} + {single_model.coef_[0]:.2f} × {selected}')
display(evaluate(single_model, [selected]).round(4))


    feature  training_R2         slope   p_value_HC3    adjusted_p  \
0      SQFT     0.477419  1.228223e+02  2.503533e-83  2.503533e-82   
1     NWKER     0.453445  4.830610e+04  1.759483e-35  1.231638e-34   
2     WKHRS     0.123825  2.571779e+05  1.630027e-61  1.304022e-60   
3     COOLP     0.010018  1.351571e+05  4.144943e-35  2.486966e-34   
4    MONUSE     0.007290  2.020521e+06  5.355188e-63  4.819669e-62   
5     HEATP     0.002499  8.508955e+04  5.712733e-06  2.856367e-05   
6  FLCEILHT     0.001592 -2.080102e+05  1.275847e-04  5.103388e-04   
7     HDD65     0.000461  3.805710e+02  9.754587e-02  2.926376e-01   
8     CDD65     0.000292 -6.647188e+02  2.287292e-01  4.574584e-01   
9   OCCUPYP     0.000017  1.773970e+04  6.015453e-01  6.015453e-01   

   significant  
0         True  
1         True  
2         True  
3         True  
4         True  
5         True  
6         True  
7        False  
8        False  
9        False  
Selected single predictor: SQFT
MFBTU = -3

# 3) Multi-variable results

Fit one multiple regression using all predictors significant in part 2. Train only on training rows; evaluate on the existing validation and test sets. MAE and RMSE are in kBtu. Marginal significance does not guarantee significance in the multiple regression. Results are unweighted associations.

In [3]:
print('Selected predictors:', ', '.join(significant_features))
multi_model = LinearRegression().fit(train[significant_features], train.MFBTU)
coefficients = pd.DataFrame({
    'variable': ['intercept'] + significant_features,
    'coefficient': [multi_model.intercept_] + multi_model.coef_.tolist()
})
print('Multiple-variable coefficients')
display(coefficients)
print('Multiple-variable performance')
display(evaluate(multi_model, significant_features).round(4))


Selected predictors: SQFT, NWKER, WKHRS, COOLP, MONUSE, HEATP, FLCEILHT
Multiple-variable coefficients
    variable   coefficient
0  intercept -1.207766e+07
1       SQFT  6.792953e+01
2      NWKER  2.702923e+04
3      WKHRS  1.207267e+05
4      COOLP  3.492580e+04
5     MONUSE -8.592510e+04
6      HEATP  1.264644e+04
7   FLCEILHT -9.966157e+04
Multiple-variable performance
                R2      MAE_kBtu     RMSE_kBtu
split                                         
train       0.5793  1.066866e+07  2.481397e+07
validation  0.6968  9.420013e+06  2.008880e+07
test        0.5496  1.145277e+07  2.817418e+07
